# Notebook 1 — Projectile Motion as a PINN (fixed launch conditions)
### My first physics-informed neural network · a research log

**What I'm doing in this project.** I want to teach a neural network to reproduce the
trajectory of a projectile *without ever showing it the answer*. Instead of training on
example points from the true curve, I'm going to train it to **obey the laws of physics**.
That kind of model is called a **Physics-Informed Neural Network (PINN)**, and this first
notebook is the simplest possible version: one fixed launch speed and one fixed angle.

**My physical setup.** I launch a projectile from the origin with speed $u$ at angle
$\theta$ and ignore air resistance for now. In a vacuum the horizontal velocity never
changes and the only acceleration is gravity pulling down. The trajectory is the familiar
parabola, and I can write the exact answer down — which is perfect for a first experiment,
because I have something to check my network against.

**My core idea (the maths I'm building on).** Because $v_x$ is constant, I can describe
the whole flight with a single second-order ODE in terms of position:
$$ \frac{d^2 y}{dx^2} \;=\; -\,\frac{g}{v_x^{2}} \;=\; \text{constant}. $$
I derive this from the parabola $y(x)=\tfrac{v_y}{v_x}x-\tfrac{g}{2v_x^2}x^2$: differentiate
once to get the slope $\tfrac{dy}{dx}=\tfrac{v_y}{v_x}-\tfrac{g}{v_x^2}x$, and again to get a
**constant** curvature $-g/v_x^2$. My PINN's job is to find a function $y(x)$ whose second
derivative equals that constant, while also starting at the origin with the right launch
slope and landing at the right place.

**My plan.** Map $x\mapsto y$ with a small network, then build a loss out of four physics
requirements (start position, launch angle, landing, and the ODE itself), and watch it
converge to the parabola. Each section below explains my reasoning before the code runs it.


## Setup — importing my tools and picking the device
I'm using **PyTorch** because it gives me *automatic differentiation* — it can compute
exact derivatives of the network's output with respect to its input. That is the single
feature that makes a PINN possible: I'll need $dy/dx$ and $d^2y/dx^2$ of the network, and
autograd hands them to me for free. I also send everything to the GPU if Colab gives me one.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## My physical constants and the quantity the network must obey
Here I fix the launch conditions ($u=30$ m/s, $\theta=30°$) and compute the things my
losses will need:
- $v_x=u\cos\theta$ and $v_y=u\sin\theta$ — the velocity components. $v_x$ stays constant
  the whole flight (no air drag), which is exactly why the ODE has a *constant* on the right.
- **Analytical range** $x_{\max}=2v_xv_y/g$ — where the parabola returns to $y=0$. I use
  this both as the right-hand edge of my domain and as a known landing point to constrain.
- **`physics_constant`** $=-g/v_x^2$ — the value $d^2y/dx^2$ must equal everywhere. This one
  number *is* my physics; the network is "correct" when its curvature matches it.

In [ ]:
g        = 9.81          # gravitational acceleration (m/s²)
v0       = 30.0          # launch speed (m/s)
theta_deg = 30.0         # launch angle (degrees)
theta    = np.radians(theta_deg)

vx = v0 * np.cos(theta)  # horizontal velocity component (constant throughout flight)
vy = v0 * np.sin(theta)  # initial vertical velocity component

# Analytical range: how far does it travel before hitting the ground?
# Setting y=0: x_max = 2*vx*vy / g
x_max_analytical = 2 * vx * vy / g
print(f"Launch speed : {v0} m/s at {theta_deg}°")
print(f"vx = {vx:.3f} m/s,  vy = {vy:.3f} m/s")
print(f"Analytical range (x at y=0): {x_max_analytical:.3f} m")

# The constant that appears in d²y/dx² = -g/vx²
# This is what the PINN's physics loss enforces
physics_constant = -g / (vx ** 2)
print(f"Physics constant d²y/dx² = {physics_constant:.6f}")

# Domain: x goes from 0 (launch) to x_max (landing)
x_range = (0.0, x_max_analytical)

## My network: a small fully-connected map $x \mapsto y$
I keep the architecture deliberately tiny — 3 hidden layers of 32 neurons — because I'm only
fitting a single smooth parabola, so I don't need much capacity.

**The one choice I will not compromise on: the activation is `Tanh`.** My loss
differentiates the network *twice*. `Tanh` is smooth and infinitely differentiable, so its
second derivative is meaningful. If I used `ReLU`, its second derivative would be zero
almost everywhere and my physics loss would carry no gradient — the PINN simply would not
learn. The output layer has **no activation**, because $y$ is an ordinary real number that
can be positive or negative and shouldn't be squashed.

In [ ]:
class FCN(nn.Module):
    """
    Fully-connected network for PINN.
    Input:  x  (horizontal position, scalar)
    Output: y  (vertical position, scalar)
    """
    def __init__(self, N_INPUT=1, N_OUTPUT=1, N_HIDDEN=32, N_LAYERS=3):
        super().__init__()
        activation = nn.Tanh        # ← NEVER change this to ReLU for a PINN
        layers = []
        # First layer: maps input (size 1) to hidden space (size 32)
        layers.append(nn.Linear(N_INPUT, N_HIDDEN))
        layers.append(activation())
        # Hidden layers
        for _ in range(N_LAYERS - 1):
            layers.append(nn.Linear(N_HIDDEN, N_HIDDEN))
            layers.append(activation())
        # Output layer: NO activation — y can be any real number (unbounded)
        layers.append(nn.Linear(N_HIDDEN, N_OUTPUT))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

pinn = FCN(N_INPUT=1, N_OUTPUT=1, N_HIDDEN=32, N_LAYERS=3).to(device)
print(pinn)
total_params = sum(p.numel() for p in pinn.parameters())
print(f"Total trainable parameters: {total_params}")

## Choosing the points where I'll enforce each condition
A PINN has three kinds of points and I set them up here:
- **Initial-condition (IC) point** at $x=0$, where I'll demand $y=0$ and the right slope.
- **Boundary-condition (BC) point** at $x=x_{\max}$, where I'll demand $y=0$ (it lands).
- **Collocation points** — 10,000 random $x$ values across the domain where I'll enforce the
  ODE. These aren't data; they're just *locations in space where physics must hold*.

The detail I want to remember: every tensor I will differentiate needs
`requires_grad=True`, otherwise autograd has nothing to track and can't give me $dy/dx$.

In [ ]:
num_ic_points          = 1     # We only have 1 initial condition point (x=0)
num_bc_points          = 1    # Boundary points at x=x_max
num_collocation_points = 10000  # Physics enforcement points (random x in domain)

# ── IC point: x=0, y=0 ──
# requires_grad=True is MANDATORY — autograd needs this to differentiate
# the network output with respect to x
x_ic = torch.tensor([[0.0]], requires_grad=True, device=device)  # shape [1,1]
y_ic_true = torch.tensor([[0.0]], device=device)                  # y(0) = 0

# ── BC points: x = x_max, y = 0 ──
x_bc = torch.full((num_bc_points, 1), x_max_analytical,
                  requires_grad=True, device=device)
y_bc_true = torch.zeros(num_bc_points, 1, device=device)         # y(x_max) = 0

# ── Collocation points: random x ∈ [0, x_max] ──
# torch.rand gives uniform random in [0,1]; scale to [0, x_max]
x_col = (torch.rand(num_collocation_points, 1, device=device)
         * x_max_analytical)
x_col.requires_grad_(True)  # needed for d²y/dx² via autograd

print(f"IC  points shape : {x_ic.shape}")
print(f"BC  points shape : {x_bc.shape}")
print(f"Col points shape : {x_col.shape}")

## The exact answer, kept aside for checking only
This is the true parabola. I am **not** training on it — the network never sees these
values. I only use it at the end to measure how close my physics-trained network got. Keeping
a known ground truth is the whole reason I started with the vacuum case.

In [ ]:
def analytical_y(x_vals):
    """True parabolic trajectory."""
    return (vy / vx) * x_vals - (g / (2 * vx**2)) * x_vals**2

x_plot = np.linspace(0, x_max_analytical, 300)
y_plot_true = analytical_y(x_plot)

## The heart of the method: turning physics into a loss
This is where the physics becomes something a network can minimise. I build four penalties,
each measuring how badly a given requirement is currently violated:

1. **IC loss** — $\big(y(0)-0\big)^2$. The launch point.
2. **Slope loss** — $\big(\tfrac{dy}{dx}(0)-\tfrac{v_y}{v_x}\big)^2$. This is how I encode the
   *launch angle*: I differentiate the network at $x=0$ and compare to the true initial slope.
3. **BC loss** — $\big(y(x_{\max})-0\big)^2$. It must come back down to the ground.
4. **Physics loss** — the residual $\big(\tfrac{d^2y}{dx^2}-(-g/v_x^2)\big)^2$ averaged over
   all collocation points. I get the second derivative by calling autograd twice, and I pass
   `create_graph=True` each time so the graph survives for the *next* differentiation and for
   backpropagation. When this residual is zero everywhere, the network is obeying my ODE.

In [ ]:
def compute_losses():
    # ── IC Loss: y(0) = 0 ──────────────────────────────────────────────────
    y_pred_ic = pinn(x_ic)
    loss_ic = torch.mean((y_pred_ic - y_ic_true) ** 2)

    # ── Slope Loss: dy/dx at x=0 = vy/vx ──────────────────────────────────
    # This enforces the launch angle.
    # grad_outputs=torch.ones_like(y_pred_ic) is needed because y_pred_ic
    # is a tensor (not scalar); this sums gradients across batch dimension.
    dy_dx_ic = torch.autograd.grad(
        y_pred_ic, x_ic,
        grad_outputs=torch.ones_like(y_pred_ic),
        create_graph=True   # ← MANDATORY: keeps graph for .backward()
    )[0]
    slope_true = torch.tensor([[vy / vx]], device=device)
    loss_slope = torch.mean((dy_dx_ic - slope_true) ** 2)

    # ── BC Loss: y(x_max) = 0 ──────────────────────────────────────────────
    y_pred_bc = pinn(x_bc)
    loss_bc = torch.mean((y_pred_bc - y_bc_true) ** 2)

    # ── Physics Loss: d²y/dx² = -g/vx² ────────────────────────────────────
    # Step 1: forward pass at collocation points
    y_col = pinn(x_col)

    # Step 2: first derivative dy/dx
    dy_dx = torch.autograd.grad(
        y_col, x_col,
        grad_outputs=torch.ones_like(y_col),
        create_graph=True   # keep graph for second differentiation below
    )[0]

    # Step 3: second derivative d²y/dx²
    d2y_dx2 = torch.autograd.grad(
        dy_dx, x_col,
        grad_outputs=torch.ones_like(dy_dx),
        create_graph=True
    )[0]

    # Step 4: residual = d²y/dx² - physics_constant  (should be 0)
    residual = d2y_dx2 - physics_constant
    loss_phys = torch.mean(residual ** 2)

    return loss_ic, loss_slope, loss_bc, loss_phys

## Training — and the reasoning behind my weights
The total loss is a weighted sum of the four terms, and the weights encode my priorities.
My reasoning:
- I push **physics** hard (`lambda_phys = 500`) because it's enforced at thousands of points
  and is the requirement that actually shapes the whole curve.
- The **launch angle** (`lambda_slope = 10`) matters more than the bare position terms,
  because if the initial slope is wrong the entire trajectory leaves at the wrong angle.
- **IC** and **BC** positions get weight 1 — they pin the two endpoints.

I use **Adam** (a robust default optimiser) and a **step scheduler** that halves the learning
rate every 2000 epochs, so I move fast early and then settle gently into the minimum instead
of bouncing around it. I also snapshot the prediction at four checkpoints so I can later
*watch* the parabola emerge.

In [ ]:
lambda_ic    = 1.0   # weight on y(0)=0 condition
lambda_slope = 10.0   # weight on dy/dx(0)=vy/vx condition (launch angle)
lambda_bc    = 1.0    # weight on y(x_max)=0 condition
lambda_phys  = 500.0    # weight on physics residual

num_epochs = 10000
optimizer = torch.optim.Adam(pinn.parameters(), lr=1e-3)

# Learning rate scheduler: decay LR after epoch 2000 to fine-tune
# This helps avoid oscillating around the minimum at late training stages
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2000, gamma=0.5)

# Bookkeeping
loss_ic_hist    = []
loss_slope_hist = []
loss_bc_hist    = []
loss_phys_hist  = []
loss_total_hist = []

# Snapshot epochs: capture the PINN's prediction at 4 training stages
checkpoints = [0, num_epochs // 4, num_epochs // 2, num_epochs - 1]
snapshots   = []   # list of dicts {'epoch': int, 'y_pred': array}

print("Starting training...")
print(f"{'Epoch':>6}  {'Total':>10}  {'IC':>10}  {'Slope':>10}  {'BC':>10}  {'Physics':>10}")

for epoch in range(num_epochs):
    optimizer.zero_grad()

    loss_ic, loss_slope, loss_bc, loss_phys = compute_losses()

    # Weighted total loss
    loss_total = (lambda_ic    * loss_ic
                + lambda_slope * loss_slope
                + lambda_bc    * loss_bc
                + lambda_phys  * loss_phys)

    # Backpropagate and update weights
    loss_total.backward()
    optimizer.step()
    scheduler.step()

    # Store history (detach from computation graph before converting to numpy)
    loss_ic_hist.append(loss_ic.detach().cpu().item())
    loss_slope_hist.append(loss_slope.detach().cpu().item())
    loss_bc_hist.append(loss_bc.detach().cpu().item())
    loss_phys_hist.append(loss_phys.detach().cpu().item())
    loss_total_hist.append(loss_total.detach().cpu().item())

    # Save snapshot at checkpoint epochs
    if epoch in checkpoints:
        pinn.eval()   # turn off dropout/batchnorm (not used here but good habit)
        with torch.no_grad():
            x_t = torch.tensor(x_plot, dtype=torch.float32,
                               device=device).unsqueeze(1)
            y_pred_np = pinn(x_t).cpu().numpy().squeeze()
        snapshots.append({'epoch': epoch, 'y_pred': y_pred_np})
        pinn.train()

    # Print progress every 500 epochs
    if epoch % 500 == 0:
        print(f"{epoch:>6}  {loss_total.item():>10.3e}  "
              f"{loss_ic.item():>10.3e}  {loss_slope.item():>10.3e}  "
              f"{loss_bc.item():>10.3e}  {loss_phys.item():>10.3e}")

print("Training complete.")


## Watching the network learn
I plot my four saved snapshots against the true parabola. At epoch 0 the prediction is
basically a random squiggle; by the final checkpoint it should sit right on top of the
analytical curve. Seeing the green start-point and blue landing-point get pinned down first,
with the curve filling in between, is a satisfying confirmation that each loss term is doing
its job.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, snap in enumerate(snapshots):
    ax = axes[idx]
    # True analytical solution
    ax.plot(x_plot, y_plot_true, 'k--', linewidth=2, label='Analytical (true)')
    # PINN prediction at this checkpoint
    ax.plot(x_plot, snap['y_pred'], 'r-', linewidth=2, label='PINN prediction')
    # Mark IC and BC points
    ax.scatter([0], [0], color='green', s=80, zorder=5, label='IC: y(0)=0')
    ax.scatter([x_max_analytical], [0], color='blue', s=80, zorder=5, label='BC: y(x_max)=0')
    ax.set_title(f"Epoch {snap['epoch']}", fontsize=13)
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_ylim(-2, y_plot_true.max() * 1.2)
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend(fontsize='small')

plt.suptitle(f"Projectile Motion PINN — v0={v0}m/s, θ={theta_deg}°", fontsize=14)
plt.tight_layout()
plt.show()

## Reading the training curves
Here I plot every loss term on a log scale. What I'm looking for: all four curves trending
downward and the physics residual dropping by several orders of magnitude. Spikes are normal
when the scheduler or the competing terms briefly fight each other; the overall trend is what
matters.

In [ ]:
epochs_arr = np.arange(num_epochs)

plt.figure(figsize=(10, 5))
plt.plot(epochs_arr, loss_ic_hist,    label='IC loss    y(0)=0',       linewidth=1.2, color='green')
plt.plot(epochs_arr, loss_slope_hist, label='Slope loss dy/dx(0)=vy/vx', linewidth=1.2, color='orange')
plt.plot(epochs_arr, loss_bc_hist,    label='BC loss    y(x_max)=0',   linewidth=1.2, color='blue')
plt.plot(epochs_arr, loss_phys_hist,  label='Physics loss d²y/dx²=const', linewidth=1.2, color='red')
plt.plot(epochs_arr, loss_total_hist, label='Total loss (weighted)',   linewidth=2.0, color='black')
plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('Loss (log scale)')
plt.title('Training Losses — Projectile Motion PINN')
plt.legend()
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

## Putting a number on the accuracy
A plot can flatter a model, so I compute hard numbers: the maximum absolute error and the
RMSE between my network and the true parabola, plus the RMSE as a percentage of the peak
height. This relative RMSE is the metric I'll reuse throughout the project to compare models
fairly.

In [ ]:
pinn.eval()
with torch.no_grad():
    x_test_t = torch.tensor(x_plot, dtype=torch.float32, device=device).unsqueeze(1)
    y_pinn_final = pinn(x_test_t).cpu().numpy().squeeze()

max_error = np.max(np.abs(y_pinn_final - y_plot_true))
rmse      = np.sqrt(np.mean((y_pinn_final - y_plot_true) ** 2))
print(f"\nFinal accuracy vs analytical solution:")
print(f"  Max absolute error : {max_error:.4f} m")
print(f"  RMSE               : {rmse:.4f} m")
print(f"  Max height (true)  : {y_plot_true.max():.4f} m")
print(f"  Relative RMSE      : {rmse / y_plot_true.max() * 100:.2f}%")

## My final trajectory, with the error shaded
One clean figure: the analytical curve, my PINN curve on top of it, and the gap between them
shaded so any residual error is visible at a glance. The endpoints are marked to show the IC
and BC really were respected.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(x_plot, y_plot_true,   'k--', linewidth=2.5, label='Analytical (true)')
plt.plot(x_plot, y_pinn_final,  'r-',  linewidth=2.0, label='PINN (final)')
plt.fill_between(x_plot,
                 y_plot_true, y_pinn_final,
                 alpha=0.2, color='red', label='Error region')
plt.scatter([0], [0], color='green', s=100, zorder=5, label='IC point')
plt.scatter([x_max_analytical], [0], color='blue', s=100, zorder=5, label='BC point')
plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.title(f"Projectile PINN — Final Result  (RMSE = {rmse:.4f} m)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## An honest stress-test: how fragile is a *fixed* PINN?
This is the question that motivates Notebook 2. I trained this network on **one** specific
$(u,\theta)$. What happens if I now ask it about slightly different launch conditions
*without retraining*? I set up helper functions to evaluate the frozen network against the
true parabola for nearby $(u,\theta)$ values. My expectation: it should degrade quickly,
because the network only ever learned a single curve.

In [ ]:
# Keep the trained model fixed
pinn.eval()

u0 = v0
theta0 = theta_deg

def analytical_y(x, u_test, theta_test_deg):
    theta_test = np.deg2rad(theta_test_deg)
    return (
        x * np.tan(theta_test)
        - (g * x**2) / (2 * u_test**2 * np.cos(theta_test)**2)
    )

def projectile_range(u_test, theta_test_deg):
    theta_test = np.deg2rad(theta_test_deg)
    return (u_test**2 * np.sin(2 * theta_test)) / g

def predict_pinn(x_values):
    with torch.no_grad():
        x_tensor = torch.tensor(
            x_values,
            dtype=torch.float32,
            device=device
        ).reshape(-1, 1)

        y_pred = pinn(x_tensor).cpu().numpy().squeeze()

    return y_pred

def test_case(u_test, theta_test_deg):
    x_max_original = projectile_range(u0, theta0)
    x_max_test = projectile_range(u_test, theta_test_deg)

    x_limit = min(x_max_original, x_max_test)
    x_eval = np.linspace(0, x_limit, 300)

    y_true = analytical_y(x_eval, u_test, theta_test_deg)
    y_pred = predict_pinn(x_eval)

    rmse = np.sqrt(np.mean((y_pred - y_true)**2))
    max_height = np.max(y_true)
    relative_rmse = (rmse / max_height) * 100

    return relative_rmse

## Mapping where the fixed model still works
I sweep $\pm10\%$ around my training $(u,\theta)$ and record the relative RMSE on a grid,
then draw it as a heatmap. I expect a small island of low error right around the point I
trained on, degrading outward. This visually demonstrates the core limitation: **a fixed PINN
does not generalise** — which is exactly the problem the generalised PINN in Notebook 2 sets
out to solve by feeding $u$ and $\theta$ in as inputs.

In [ ]:
u_values = np.linspace(0.90 * u0, 1.10 * u0, 21)
theta_values = np.linspace(0.90 * theta0, 1.10 * theta0, 21)

error_grid = np.zeros((len(theta_values), len(u_values)))

for i, theta_test in enumerate(theta_values):
    for j, u_test in enumerate(u_values):
        error_grid[i, j] = test_case(u_test, theta_test)

plt.figure(figsize=(8, 6))
plt.imshow(
    error_grid,
    origin="lower",
    extent=[u_values[0], u_values[-1], theta_values[0], theta_values[-1]],
    aspect="auto",
    cmap="viridis"
)
plt.colorbar(label="Relative RMSE (%)")
plt.xlabel("u")
plt.ylabel("theta")
plt.title("Fixed PINN Sensitivity Without Retraining")
plt.show()

## Quantifying the usable region
Finally I threshold the grid at 5% relative RMSE and print the $(u,\theta)$ range that stays
acceptable. The narrowness of this region is the punchline of Notebook 1 and my motivation to
generalise next.

In [ ]:
threshold = 5.0

valid = error_grid <= threshold

valid_indices = np.argwhere(valid)

if len(valid_indices) > 0:
    valid_u = []
    valid_theta = []

    for i, j in valid_indices:
        valid_theta.append(theta_values[i])
        valid_u.append(u_values[j])

    print("Acceptable error threshold:", threshold, "%")
    print("Valid u range:", min(valid_u), "to", max(valid_u))
    print("Valid theta range:", min(valid_theta), "to", max(valid_theta))
else:
    print("No valid region found under threshold.")

## What I take away from Notebook 1
- I trained a network to match a parabola using **only physics**, no trajectory data.
- The key machinery was **autograd** (for $d^2y/dx^2$) and a **`Tanh`** network (so those
  derivatives exist), with a weighted multi-term loss balancing IC, launch angle, BC, and ODE.
- The big limitation I measured directly: this model only knows the *one* launch it trained
  on. **Next (Notebook 2):** I feed $u$ and $\theta$ in as extra inputs so a single network
  can cover a whole family of launches.